# Классификация обзоров на  игру CS2 из Steam с использованием BERT

В качестве метки взята бинарная метка "Recommended"/"Not Recommended". На сайте Steam пользователь сам ее выбирает при написании обзора.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import pandas as pd
df = pd.read_csv('/content/drive/MyDrive/CS_BERT_cl/CS2_reviews_done_ok.csv', sep =',', header = 0)
df.head()

,rating,comm,date,playtime,comm_ready,rating_bool
0,Recommended,"Топ игра, никому не советую.",Posted: 21 Apr @ 4:18am,"95.7 hrs last two weeks / 1,711.7 hrs on recor...",топ игра никто советовать,1
1,Not Recommended,какашка ыыыыы,Posted: 21 Apr @ 4:14am,2.8 hrs last two weeks / 48.8 hrs on record(46...,какашка ыыыы,0
2,Recommended,крутая игра,Posted: 21 Apr @ 4:10am,19.0 hrs last two weeks / 32.4 hrs on record(3...,крутой игра,1
3,Not Recommended,"игра , дофига читаков, тупые тимейты, нет анти...",Posted: 21 Apr @ 4:11am,87.5 hrs last two weeks / 301.1 hrs on record(...,игра дофига читаковы тупой тимейт античита раб...,0
4,Recommended,оа мтв аиб,Posted: 21 Apr @ 4:06am,0.0 hrs last two weeks / 291.2 hrs on record,оа мтв аиб,1


In [ ]:
# небольшой препроцессинг
import re
for i in range(len(df['rating'])): # удаляем какие-то юникоды
  df['rating'].loc[i] = re.sub(r'<U\+[\w\d]+>', ' ', df['rating'].loc[i])
  df['rating'].loc[i] = re.sub(' +', ' ', df['rating'].loc[i]).strip()

Выходные данные были обрезаны до нескольких последних строк (5000).
See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

  df['rating'].loc[i] = re.sub(' +', ' ', df['rating'].loc[i]).strip()
/tmp/ipython-input-1047947447.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['rating'].loc[i] = re.sub(' +', ' ', df['rating'].loc[i]).strip()
/tmp/ipython-input-1047947447.py:4: FutureWarning: ChainedAssignmentError: behaviour will change in pandas 3.0!
You are setting values through chained assignment. Currently this works in certain cases, but when using Copy-on-Write (which will become the default behaviour in pandas 3.0) this will never work to update the original DataFrame or Series, because the intermediate object

In [ ]:
# надо поменять лейбелы на циферки
df['rating_bool'] = df['rating']
for i in range(len(df)):
  if df['rating'].loc[i] == 'Not Recommended':
    df['rating_bool'].loc[i] = 0
  else:
    df['rating_bool'].loc[i] = 1
df.head()


/tmp/ipython-input-3403166054.py:7: FutureWarning: ChainedAssignmentError: behaviour will change in pandas 3.0!
You are setting values through chained assignment. Currently this works in certain cases, but when using Copy-on-Write (which will become the default behaviour in pandas 3.0) this will never work to update the original DataFrame or Series, because the intermediate object on which we are setting values will behave as a copy.
A typical example is when you are setting values in a column of a DataFrame, like:

df["col"][row_indexer] = value

Use `df.loc[row_indexer, "col"] = values` instead, to perform the assignment in a single step and ensure this keeps updating the original `df`.

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

  df['rating_bool'].loc[i] = 1
/tmp/ipython-input-3403166054.py:5: FutureWarning: ChainedAssignmentError: behaviour will change in pandas 3.0!
You are setting va

,rating,comm,date,playtime,comm_ready,rating_bool
0,Recommended,"Топ игра, никому не советую.",Posted: 21 Apr @ 4:18am,"95.7 hrs last two weeks / 1,711.7 hrs on recor...",топ игра никто советовать,1
1,Not Recommended,какашка ыыыыы,Posted: 21 Apr @ 4:14am,2.8 hrs last two weeks / 48.8 hrs on record(46...,какашка ыыыы,0
2,Recommended,крутая игра,Posted: 21 Apr @ 4:10am,19.0 hrs last two weeks / 32.4 hrs on record(3...,крутой игра,1
3,Not Recommended,"игра , дофига читаков, тупые тимейты, нет анти...",Posted: 21 Apr @ 4:11am,87.5 hrs last two weeks / 301.1 hrs on record(...,игра дофига читаковы тупой тимейт античита раб...,0
4,Recommended,оа мтв аиб,Posted: 21 Apr @ 4:06am,0.0 hrs last two weeks / 291.2 hrs on record,оа мтв аиб,1


In [ ]:
df.to_csv('/content/drive/MyDrive/CS_BERT_cl/CS2_reviews_done_ok.csv',index=False)

In [ ]:
df['rating'].value_counts()
# видно что сильно много положительных рекомендаций

,count
rating,
Recommended,3445
Not Recommended,1346


In [ ]:
# уменьшим выборку Recommended
import random
import numpy as np
all_indices = df[df['rating'] == 'Recommended'].index
indices_to_drop = np.random.choice(all_indices, size=2048, replace=False)
df.drop(indices_to_drop, inplace=True)
df['rating'].value_counts()

,count
rating,
Recommended,1397
Not Recommended,1346


In [ ]:
#сохраним получившееся на всякий случай
df.to_csv('/content/drive/MyDrive/CS_BERT_cl/CS2_reviews_done_ok_undersamlpled.csv',index=False)

In [ ]:
# щас сделаем класс нашего датасета
import torch
from transformers import AutoTokenizer
from torch.utils.data import Dataset
class TextDataset(Dataset):
    def __init__(self, dataframe, max_length=256, tokenizer_name='bert-base-multilingual-cased'):
        self.texts = dataframe['comm'].values # нумпай вектор с текстом получаем
        self.targets = dataframe['rating_bool'].values # нумпай вектор с токенами получаем
        self.tokenizer = AutoTokenizer.from_pretrained(tokenizer_name) # токенизатор определяем
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = self.texts[idx]
        target = self.targets[idx]
        encoding = self.tokenizer.encode_plus(
            text,
            add_special_tokens=True,
            max_length=self.max_length,
            padding='max_length',
            truncation=True,
            return_attention_mask=True,
            return_tensors='pt'
        )
        input_ids = encoding['input_ids'].flatten()
        attention_mask = encoding['attention_mask'].flatten()

        return {
            'input_ids': torch.as_tensor(input_ids, dtype=torch.long),
            'attention_mask': torch.as_tensor(attention_mask, dtype=torch.long),
            'ratings': torch.as_tensor(target, dtype=torch.long),
            'comms': text
        }

In [ ]:
# разделяй и обучай
from sklearn.model_selection import train_test_split
train_data, test_data = train_test_split(df, train_size=0.8, shuffle=True, random_state=42)
print(f'Train size: {len(train_data)}', f'Test size: {len(test_data)}', sep = '\n')

Train size: 2194
Test size: 549


In [ ]:
# делаем инстансы датасета
train_dataset = TextDataset(train_data)
test_dataset = TextDataset(test_data)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/625 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/996k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.96M [00:00<?, ?B/s]

In [ ]:
# сделаем класс загрузчика данных, чтобы во время дообучения не надо было весь датасет подгружать сразу
from torch.utils.data import DataLoader

BATCH_SIZE = 16
torch.manual_seed(42)
train_loader = DataLoader(train_dataset,
                          batch_size=BATCH_SIZE,
                          shuffle=True)
test_loader = DataLoader(test_dataset,
                         batch_size=len(test_dataset))
next(iter(train_loader))

{'input_ids': tensor([[  101, 19732, 65412,  ...,     0,     0,     0],
         [  101, 14440, 10241,  ...,   119,   119,   102],
         [  101, 10375, 29140,  ...,     0,     0,     0],
         ...,
         [  101, 10297, 15469,  ...,     0,     0,     0],
         [  101,   556, 17971,  ...,     0,     0,     0],
         [  101, 45784, 11092,  ...,     0,     0,     0]]),
 'attention_mask': tensor([[1, 1, 1,  ..., 0, 0, 0],
         [1, 1, 1,  ..., 1, 1, 1],
         [1, 1, 1,  ..., 0, 0, 0],
         ...,
         [1, 1, 1,  ..., 0, 0, 0],
         [1, 1, 1,  ..., 0, 0, 0],
         [1, 1, 1,  ..., 0, 0, 0]]),
 'ratings': tensor([1, 1, 0, 0, 1, 1, 0, 0, 0, 1, 0, 1, 0, 0, 1, 0]),
 'comms': ['Норм игра после фф',
  'ммм........................................................................................................................................................................................................................................................................

In [ ]:
# настраиваем девайс для обучения
from torch import cuda
torch.cuda.empty_cache()
device = 'cuda' if cuda.is_available() else 'cpu'

In [ ]:
from transformers import BertForSequenceClassification, BertConfig
from torch.optim import AdamW

model = BertForSequenceClassification.from_pretrained(
    "bert-base-multilingual-cased",
    num_labels = 2,
    output_attentions = False,
    output_hidden_states = False,
)

model.to(device)

model.safetensors:   0%|          | 0.00/714M [00:00<?, ?B/s]

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-multilingual-cased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


BertForSequenceClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(119547, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSdpaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1

In [ ]:
# Гиперпараметры
# batch size уже задавали, он 16
EPOCHS = 6

optimizer = AdamW(model.parameters(),
                  lr = 1e-5,
                  eps = 1e-8)

In [ ]:
import time
import datetime
import numpy as np
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

In [ ]:
# Надо куду настроить, а то памяти не хватает
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
print(f"PYTORCH_CUDA_ALLOC_CONF is set to: {os.environ.get('PYTORCH_CUDA_ALLOC_CONF')}")

PYTORCH_CUDA_ALLOC_CONF is set to: expandable_segments:True


In [ ]:
def save_checkpoint(epoch, model, optimizer, loss, filepath):
    torch.save({
        'epoch': epoch,
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'loss': loss,
    }, filepath)
    print(f"Checkpoint saved at epoch {epoch} to {filepath}")

In [ ]:
# готовим к трене
training_stats = []
epoch_loss_train = []
total_t0 = time.time()

# TRAINING
for epoch in range(1, EPOCHS + 1):
    model.train()
    t0 = time.time()
    print("")
    print("================ Epoch {:} / {:} ================".format(epoch, EPOCHS))
    train_all_predictions = []
    train_all_true_labels = []
    for step, data in enumerate(train_loader):
        if step % 40 == 0 and not step == 0:
            elapsed = int(round(time.time() - t0))
            elapsed = str(datetime.timedelta(seconds=elapsed))
            print(
                "  Batch {:>5,}  of  {:>5,}.    Elapsed: {:}.".format(
                    step, len(train_loader), elapsed
                )
            )

        targets = data["ratings"].to(device)
        mask = data["attention_mask"].to(device)
        ids = data["input_ids"].to(device)

        model.zero_grad()

        loss, logits = model(
            ids, token_type_ids=None, attention_mask=mask, labels=targets
        ).to_tuple()
        epoch_loss_train.append(loss.item())

        cpu_logits = logits.cpu().detach().numpy()
        train_all_predictions.extend(np.argmax(cpu_logits, axis=1).flatten())
        train_all_true_labels.extend(targets.cpu().numpy())

        loss.backward()
        optimizer.step()
    train_accuracy = accuracy_score(train_all_true_labels, train_all_predictions)
    train_precision, train_recall, train_f1, _ = precision_recall_fscore_support(
        train_all_true_labels, train_all_predictions, average="binary"
    )
    print("")
    print('---TRAIN METRICS---')
    print(f"Loss: {np.mean(epoch_loss_train):.4f}")
    print(f"Accuracy: {train_accuracy:.4f}")
    print(f"Precision: {train_precision:.4f}")
    print(f"Recall: {train_recall:.4f}")
    print(f"F1-Score: {train_f1:.4f}")
    print("")
    #save_model
    state = {
        'epoch': epoch,
        'state_dict': model.state_dict(),
        'optimizer': optimizer.state_dict(),
    }
    save_checkpoint(epoch, model, optimizer, loss, f'/content/checkpoint{epoch}.pth')


================ Epoch 1 / 6 ================
  Batch    40  of    138.    Elapsed: 0:00:26.
  Batch    80  of    138.    Elapsed: 0:00:51.
  Batch   120  of    138.    Elapsed: 0:01:17.

---TRAIN METRICS---
Loss: 0.6375
Accuracy: 0.6249
Precision: 0.6391
Recall: 0.5971
F1-Score: 0.6174

Checkpoint saved at epoch 1 to /content/checkpoint1.pth

================ Epoch 2 / 6 ================
  Batch    40  of    138.    Elapsed: 0:00:26.
  Batch    80  of    138.    Elapsed: 0:00:53.
  Batch   120  of    138.    Elapsed: 0:01:19.

---TRAIN METRICS---
Loss: 0.5847
Accuracy: 0.7507
Precision: 0.7618
Recall: 0.7392
F1-Score: 0.7503

Checkpoint saved at epoch 2 to /content/checkpoint2.pth

================ Epoch 3 / 6 ================
  Batch    40  of    138.    Elapsed: 0:00:26.
  Batch    80  of    138.    Elapsed: 0:00:53.
  Batch   120  of    138.    Elapsed: 0:01:20.

---TRAIN METRICS---
Loss: 0.5314
Accuracy: 0.8200
Precision: 0.8256
Recall: 0.8174
F1-Score: 0.8215

Checkpoint saved a

In [ ]:
#!pip install numba
from numba import cuda


device = cuda.get_current_device()
device.reset()

In [ ]:
# Тестирование (на раннее сохраненных чекпоинтах)
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
for epoch in range(1, EPOCHS+1):
    # loading model
    model = BertForSequenceClassification.from_pretrained(
    "bert-base-multilingual-cased",
    num_labels = 2,
    output_attentions = False,
    output_hidden_states = False,
)
    checkpoint = torch.load(f"/content/checkpoint{epoch}.pth", map_location=torch.device(device))
    model.load_state_dict(checkpoint['model_state_dict'])
    model.to(device)
    # VALIDATION
    print("Running validation ...")
    print("")
    model.eval()
    epoch_loss_test = []
    test_all_predictions = []
    test_all_true_labels = []
    for data in test_loader:
        targets = data["ratings"].to(device)
        mask = data["attention_mask"].to(device)
        ids = data["input_ids"].to(device)

        with torch.no_grad():
            loss, logits = model(ids, token_type_ids=None, attention_mask=mask, labels=targets).to_tuple()

        epoch_loss_test.append(loss.item())
        cpu_logits = logits.cpu().detach().numpy()
        test_all_predictions.extend(np.argmax(cpu_logits, axis=1).flatten())
        test_all_true_labels.extend(targets.cpu().numpy())
    test_accuracy = accuracy_score(test_all_true_labels, test_all_predictions)
    test_precision, test_recall, test_f1, _ = precision_recall_fscore_support(
        test_all_true_labels, test_all_predictions, average="binary"
    )
    print("")
    print('---TEST METRICS---')
    print(f"Loss: {np.mean(epoch_loss_test):.4f}")
    print(f"Accuracy: {test_accuracy:.4f}")
    print(f"Precision: {test_precision:.4f}")
    print(f"Recall: {test_recall:.4f}")
    print(f"F1-Score: {test_f1:.4f}")

    training_stats.append(
            {
            'epoch': epoch,
            'Training Loss': np.mean(epoch_loss_train),
            'Training Accuracy': train_accuracy,
            'Training Precision': train_precision,
            'Training Recall': train_recall,
            'Training F1': train_f1,
            'Validation Loss': np.mean(epoch_loss_test),
            'Validation Accuracy': test_accuracy,
            'Validation Precision': test_precision,
            'Validation Recall': test_recall,
            'Validation F1': test_f1
        }
    )

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-multilingual-cased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Running validation ...


---TEST METRICS---
Loss: 0.5943
Accuracy: 0.6831
Precision: 0.7761
Recall: 0.5474
F1-Score: 0.6420


Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-multilingual-cased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Running validation ...


---TEST METRICS---
Loss: 0.5267
Accuracy: 0.7486
Precision: 0.7155
Recall: 0.8561
F1-Score: 0.7796


Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-multilingual-cased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


...

Running validation ...


---TEST METRICS---
Loss: 0.5489
Accuracy: 0.7486
Precision: 0.8387
Recall: 0.6386
F1-Score: 0.7251


Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-multilingual-cased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Running validation ...


---TEST METRICS---
Loss: 0.5582
Accuracy: 0.7687
Precision: 0.7862
Recall: 0.7614
F1-Score: 0.7736


Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-multilingual-cased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Running validation ...


---TEST METRICS---
Loss: 0.6937
Accuracy: 0.7596
Precision: 0.8122
Recall: 0.6982
F1-Score: 0.7509


Таким образом, удалось достичь Accuracy ~0.75. Хотя можно заметить, что Recall очень сильно скачет вверх-вниз в зависимости от эпохи обучения, это может быть связано с особенностью поведения сообщества игры. И значительная часть пользователей оставляют обзоры, в которых содержание не совпадает с выбранной пользователем меткой ("Recommended"/"Not Recommended"), не воспринимая игру вполне всерьез.